<a href="https://colab.research.google.com/github/studiousdoston/computer_vision/blob/main/data_scraping.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q "requests==2.32.4" "pillow<13"

In [ ]:
import os, io, time, hashlib, requests
from ddgs import DDGS
from PIL import Image
from google.colab import drive

In [ ]:
drive.mount("/content/drive", force_remount=True)
DATASET_DIR = "/content/drive/MyDrive/food101_dataset"

CLASSES = {
    "pilaf": ["uzbek plov dish", "osh plov rice carrots", "plov in kazan"],
    "kebab": ["shashlik skewers grilled", "uzbek shashlik meat", "shashlik on plate"],
}
IMAGES_PER_CLASS = 100

Mounted at /content/drive


In [ ]:
def search_urls(queries, max_per_query=80):
    urls = []
    with DDGS() as ddgs:
        for q in queries:
            try:
                results = ddgs.images(
                    q,
                    region="us-en",
                    safesearch="on",
                    type_image="photo",
                    max_results=max_per_query,
                )
                urls += [r["image"] for r in results]
            except Exception as e:
                print(f"  search failed for '{q}': {e}")
            time.sleep(2)
    return list(dict.fromkeys(urls))

In [ ]:
def download_valid_image(url):
    try:
        r = requests.get(url, timeout=10, headers={"User-Agent": "Mozilla/5.0"})
        r.raise_for_status()
        img = Image.open(io.BytesIO(r.content))
        img.verify()
        img = Image.open(io.BytesIO(r.content)).convert("RGB")
        if min(img.size) < 150:
            return None, None
        return img, hashlib.md5(r.content).hexdigest()
    except Exception:
        return None, None


In [ ]:
for folder, queries in CLASSES.items():
    print(f"\n[{folder}] searching...")
    target = os.path.join(DATASET_DIR, folder)
    os.makedirs(target, exist_ok=True)
    urls = search_urls(queries)
    print(f"  found {len(urls)} candidate URLs")
    # Create the class folder on Drive and gather candidate URLs

    seen, saved = set(), 0
    for url in urls:
        if saved >= IMAGES_PER_CLASS:
            break
        img, h = download_valid_image(url)
        if img is None or h in seen:
            continue
        seen.add(h)
        img.save(os.path.join(target, f"{folder}_{saved:03d}.jpg"), "JPEG", quality=90)
        saved += 1
    print(f"  saved {saved} images to {target}")
    # Downloads until 100 good images are saved, skipping broken files and
    # exact duplicates, and names them pilaf_000.jpg, pilaf_001.jpg, and so on

drive.flush_and_unmount()
print("\nDone. Drive synced.")
# Forces Colab to finish writing everything to Drive before you check it


[pilaf] searching...
  found 103 candidate URLs
  saved 100 images to /content/drive/MyDrive/food101_dataset/pilaf

[kebab] searching...
  found 104 candidate URLs
  saved 100 images to /content/drive/MyDrive/food101_dataset/kebab

Done. Drive synced.
